### Module 1 Project

In the first 6 Chapters I learned:

- Variables and data types
- Basic operations
- Input and output
- Type conversion
- Lists and tuples
- Dictionaries and sets
- Functions
- Control flow (if statements, loops)
- Basic File Operations

Using this knowledge, I want to build the first part of my Python Portfolio.<br>
The Portfolio itself should eventually become a flexible tool to automate tasks within a machine learning workflow. <br>The first part consists of:

- Sourcing Subroutine:  <br>Tha application should read a text file containing a list of urls, the application should iterate through the list and download the files from the internet or a given path, into the sourcing folder.  <br> At the end of the preocess, the application should print out a summary of every step. (url / path, timestamp,  filename, size, time taken to download).  <br>For the failed downloads, the user can choose wehther to create a separate list of the config-file for another attempt. It should ask whether the user wants to retry the failed downloads before next initiation with the original list starts, should the user chooses no.

- Data Load Subroutine: <br> The application should read a text file containing lists of file selection (file name filter), the cleansing function(s), it should apply to the files. The cleaned version should be saved in a separate directory. A summary should be printed out for every operation done for each file.

- Analyze Data Subroutine:  <br>The user can choose from a list of files (clean data) to display different basic statistics of the data. And the end of the operation, the user can choose whether to choose another file to look at or quit.
 <br>



## Notebook initialization
Initiate global variables + import libraries

In [ ]:
import urllib.request
import shutil
import os
import pandas as pd
import numpy as np
from scipy import stats
import functools
from time import time
import matplotlib as plt

# Constants
global WORKING_DIRECTORY, SOURCE_DIRECTORY, PREPROCESS_DIRECTORY, TESTS_DIRECTORY,OUTPUT_DIRECTORY, RAW_DATAFILES, SUPPORTED_FILEFORMATS, LOG_CALL

# Get the current working directory
WORKING_DIRECTORY = os.getcwd()
print("The current working directory is:", WORKING_DIRECTORY)

# Set subdirectories
SOURCE_DIRECTORY = WORKING_DIRECTORY+ "\\2_Sourcing\\"
PREPROCESS_DIRECTORY = WORKING_DIRECTORY+ "\\3_Data_Preprocessing\\"
TESTS_DIRECTORY = WORKING_DIRECTORY+ "\\4_Tests\\"
OUTPUT_DIRECTORY = WORKING_DIRECTORY+ "\\8_Output\\"

# Dictionary of supported fileformats and the associated function to read it
SUPPORTED_FILEFORMATS={"CSV":pd.read_csv,"TXT":pd.read_fwf,"JSON":pd.read_json,"XML":pd.read_xml,"XLSX":pd.read_excel,"XLS":pd.read_excel}

#Raw Data Files {key=filename,value=pd.dataframe}
RAWDATAFILES={}

#Log function calls (boolean)
LOG_CALL=False

### Log_calls Function
This function provide the functionality to easily log a function call by decorating the target function, 
providing duration, argument and return in the output console.

In [ ]:
def log_calls(func):
###
# Apply writing @log_calls before the function to decorate a function
###
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        global LOG_CALL
        if LOG_CALL:
            print(f"calling {func.__name__} with args: {args}, kwargs: {kwargs}")
            time_start=time()
            result = func(*args,**kwargs)
            time_end=time()
            print(f"func:{func.__name__} args:[{args}, {kwargs}] took: {format(int((time_end-time_start)))} sec")
            print(f"{func.__name__} returned: {result}")
        else:
            result = func(*args,**kwargs)
        return result
    return wrapper

### Read Config Function
Should read each line from a cfg file (text) and store it in a list.

In [ ]:
def read_config(file_path,targetlist):
###
# Arguments:
# file_path: Location of the editable source.cfg file, 1 line per file in the following format [url / folder path],[filename],[target-folder]
# targetlist: can be sourceCFGList or missCFGList. Format: item [url, filename, destination]
###
    try:
        with open(file_path, 'r') as file:
            config = file.read()
            for line in config.splitlines():
                #print(line.strip())
                x,y,z=line.strip().split(',')
                targetlist.append([x,y,z])
            print("Configuration loaded:")
            print(config)
    except FileNotFoundError:
        print(f"Configuration file '{file_path}' not found.")
        return None

### Write Config Function
Writes back the missing files into the miss.cfg file for the next run.

In [ ]:
def write_config(file_path: str, data: list):
###
# Arguments: 
# file_path (STR): Location of the editable source.cfg file, 1 line per file in the following format [url / folder path],[filename],[target-folder]
# data: list of sourcable items to store. Format: item [url, filename, destination]
###
    try:
        with open(file_path, 'w') as file:
            for line in data:
                file.write(','.join(line) + '\n')
        print(f"Configuration saved to '{file_path}'")
    except Exception as e:
        print(f"Error writing configuration file: {e}")
        return None

### Download File
Retrieve a file from www and save it to the destination path

In [ ]:
@log_calls
def download_file(url, filename, destination):
    """
    # Arguments
    # url: http:// address of the the file
    # filename: name of the retrieved file
    # destination: path of the target folder (unc format)
    """
    ## exception handling moved to upper layer
    #try:
    urllib.request.urlretrieve(url, WORKING_DIRECTORY + destination)
    print(f"File downloaded successfully: {destination}")
    #except Exception as e:
    #    print(f"Error downloading file: {e}")
    #    return None


### Copy File Function

Copy file from UNC-Path and save it to destination.

In [116]:
@log_calls
def copy_file(source, destination):
    """
    Arguments
    source: unc path of the file to be copied
    destination: path of the target folder (unc format)
    """
    ## exception handling moved to upper layer
    #try:
    shutil.copy(source, WORKING_DIRECTORY+ destination)
    print(f"File copied successfully from {source} to {destination}")
    #except Exception as e:
    #   print(f"Error copying file: {e}")

## Sourcing Subroutine
Initialize the Sourcing Subroutine by importing files from different sources (web or network share) using a predefined config-file.<br>
If missing files from last iteration exists, automatically ask the user whether to continue with the import or retry missing files first.

In [ ]:
def sourcing_process():
    #Source Items
    sourceCFG = SOURCE_DIRECTORY + "source.cfg"
    sourceCFGList = []

    #Missing items
    missCFG = SOURCE_DIRECTORY +"miss.cfg"
    missCFGList = []

    print("You chose Sourcing")

    def source_list(CFGList):
        """
        Iterate through a list and call download or copy file function according to the given sourcepath and call the write cfg for failed attempts.
        Parameter list of Array ( URL, Filename, Destination)
        """
        NewmissCFGList = []
        for sourceCFG in CFGList:
            url = str(sourceCFG[0])
            filename = sourceCFG[1]
            destination = sourceCFG[2]
            try:
                if url.lower().startswith("http"):
                    download_file(url, filename, destination)
                    #missCFGList.remove[sourceCFG] does not work while in the for loop
                else:
                    copy_file(url, destination)
                    #missCFGList.remove[sourceCFG] dito
            except Exception as e:
                print(f"Error sourcing file: {e}")
                NewmissCFGList.append([url,filename,destination])
        CFGList=[]
        if NewmissCFGList:
            write_config(missCFG, NewmissCFGList)
            raise RuntimeError("Not all items were sourced." + str(NewmissCFGList).join("\n"))

    #precheck whether missing items / file (bkp.cfg) exists
    if os.path.exists(missCFG):
        read_config(missCFG,missCFGList)
        #check whether list exists / not empty
        if missCFGList:
            response = input("There are open Items for download, do you whish to retry (y/n)?\n" + str(missCFGList).join("\n")).strip().lower()
            if response == 'y':
                try:
                    #Source missing items
                    source_list(missCFGList)

                    #The following will be skipped if an error occures in the above function:
                    os.remove(missCFG)
                except Exception as e:
                    print("Retry unsuccessful.")
            else:
                response =input("Do you want to delete the open itmes (y/n)?")
                if response =='y':
                    os.remove(missCFG)
    
    #main sourcing process
    if os.path.exists(sourceCFG):
        response = input("Would you like to start sourcing the data? (y/n): ").strip().lower()
        if response == 'y':
            print("Starting data sourcing...")
            read_config(sourceCFG,sourceCFGList)
            if sourceCFGList:
                #Source all files listed in config
                source_list(sourceCFGList)
    else:
        print(f"Source configuration file '{sourceCFG}' not found.")

### Load File function
Load File into memory for processing.

In [ ]:
@log_calls
def loadfile(filename):
    """
    Arguments
    filename: must be full unc path
    """
    try:
        global RAWDATAFILES

        base_name = os.path.basename(filename)       

        #Choose respective reader according to filetype
        df=SUPPORTED_FILEFORMATS[filename.upper().split(".")[-1]](filename)
        df.info()

        #Data cleansing (not parameterized / automated yet) only for excel
        if filename.upper().split(".")[-1] =='XLSX':
            #Reload skipping header and footer
            df=pd.read_excel(filename,skiprows=5,nrows=66912)
            df.columns = ['firstname','female','male']
            
            #resolve issue with strings in data-area
            df.iloc[:,1:3]=df.iloc[:,1:3].replace("*",0)
            df['male'] = pd.to_numeric(df['male'],downcast='integer',errors='coerce').fillna(0).astype(int)
            df['female'] = pd.to_numeric(df['female'],downcast='integer',errors='coerce').fillna(0).astype(int)
        
        #Append the data frame to the global dictionary for further usage
        RAWDATAFILES[base_name] = df.copy()
        print(f"File {base_name} successfully loaded:")
        
    except Exception as e:  
        print(f"Could not load file: {filename}. Error: {e}") 


### Analyzefile Function
This function is to provide additional information about a data frame<br>

In [ ]:
@log_calls
def analyzefile(filename):
    """
    # Argmunets
    # filename: must be full unc path (file must be already loaded and available in RAWDATAFILES)
    """
    global RAWDATAFILES
    df=RAWDATAFILES[filename]

    #print metadata
    #print(df)
    #print("head:")
    #print(df.head())
    #print("tail:")
    #print(df.tail())
    
    print("describe:")
    print(df.describe())
    print("columns:")
    print(df.columns)
    print("index:")
    print(df.index)

    if filename.upper().split(".")[-1] =='XLSX':
        #Analyze Excel Firstname Ranking 2024
        #for i in range(ord('a'), ord('z')+1):
            #print(chr(i))

        # module 2
        for column in df.columns:               
                #analze outliers using z-Score, z-Score = (row value - mean of column)/standard deviation of column
                if column=="female":
                    df_outlier=df[df[column]>0].copy()
                    df_outlier['f_zscore'] =np.abs(stats.zscore(df_outlier[column]))
                    print(df_outlier.sort_values(column,ascending=False))

                if column=="male":
                    df_outlier=df[df[column]>0].copy()
                    df_outlier['m_zscore'] =np.abs(stats.zscore(df_outlier[column]))
                    print(df_outlier.sort_values(column,ascending=False))

## Data Load Subroutine
Initialize the data load subroutine by showing a list of available files and load it upon user's choice.

In [ ]:
def dataload_process():
    print("You chose Data Load")
    
    #get all files from source directory
    rawlist = os.listdir(SOURCE_DIRECTORY)
    #filter non data files (list comprehension)
    filteredlist = [x for x in rawlist if x.upper().split(".")[-1] in list(SUPPORTED_FILEFORMATS.keys())]
    
    possibleChoice =[]

    print("Please choose one of the following files to be loaded: ")
    if filteredlist:
        for idx, file in enumerate(filteredlist):
            print(f"{idx}: {file}")
            possibleChoice.append(str(idx))

        possibleChoice.append("x")
        print("x: (abort)")
    else:
        print('Please source data in Step 1 first.')
        return

    inputMissing = True
    while inputMissing:
        response = input("Please input the number of the choosen file:")
        if response in possibleChoice:
            inputMissing=False
    
    if not response =="x":
        loadfile(SOURCE_DIRECTORY + filteredlist[int(response)])
        inputMissing = True
        while inputMissing:
            response = input("Do you want to load another file? (y/n)")
            if response =='y':
                dataload_process()
                inputMissing=False
            if response=='n':
                inputMissing=False
                return
        

## Analyze Data Subroutine
Initiate the analyze data subroutine by showing the user a list of available dataframes and  asking which file to analyze.

In [ ]:
def analyzedata_process():
    global RAWDATAFILES
    possibleChoice =[]

    if RAWDATAFILES:
        for idx, file in enumerate(RAWDATAFILES):
            print(f"{idx}: {file}")
            possibleChoice.append(str(idx))

        possibleChoice.append("x")
        print("x: (abort)")
        
        inputMissing = True
        while inputMissing:
            response = input("Please input the number of the choosen file:")
            if response in possibleChoice:
                inputMissing=False
        
        if not response =="x":
            analyzefile(file)
            inputMissing = True
            while inputMissing:
                response = input("Do you want to analyze another file? (y/n)")
                if response =='y':
                    analyzedata_process()
                    inputMissing=False
                if response=='n':
                    inputMissing=False
                    return
    else:
        print('Please load data in Step 2 first.')
        

# Main UX Loop

Iterate through the main processes with user prompts

In [ ]:
if __name__ == "__main__":
    exit = False
    while exit != True:
        response = input("Please choose processing mode: \n 1. Sourcing \n 2. Load Data \n 3. Analyze Data\n " \
        "4. Results (not implemented yet)\n 5. Walkthrough (not implemented yet)\n l(switch log_calls) \n x (exit)")
        if response == '1':
           sourcing_process()
        elif response == '2':
            dataload_process()
        elif response == '3':
            analyzedata_process()
        elif response == 'l':
            global LOG_CALL
            LOG_CALL= not LOG_CALL
            print(f"Log_call: {LOG_CALL}")
        elif response == 'x':
            exit = True
            print("Exiting the program.")